In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [3]:
model_client = OpenAIChatCompletionClient(
    model='gpt-4o-mini',
    temperature=0.3
)

In [4]:
# Note: This example uses mock tools instead of real APIs for demonstration purposes
def search_web_tool(query: str) -> str:
    if "2006-2007" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."


def percentage_change_tool(start: float, end: float) -> float:
    return ((end - start) / start) * 100

In [5]:

planning_agent = AssistantAgent(
    "PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

web_search_agent = AssistantAgent(
    "WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[search_web_tool],
    model_client=model_client,
    system_message="""
    You are a web search agent.
    Your only tool is search_tool - use it to find information.
    You make only one search call at a time.
    Once you have the results, you never do calculations based on them.
    """,
)

data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""
    You are a data analyst.
    Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided.
    If you have not seen the data, ask for it.
    """,
)

In [6]:
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

In [8]:
text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=25)
termination = text_mention_termination | max_messages_termination

In [9]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [10]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console

In [12]:
team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,  # Allow an agent to speak multiple turns in a row.
)

In [13]:
task = """
Who was the Miami Heat player with the highest points in the 2006-2007 season, 
and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
"""

# Use asyncio.run(...) if you are running this in a script.
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------

Who was the Miami Heat player with the highest points in the 2006-2007 season, 
and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?

---------- TextMessage (PlanningAgent) ----------
To address your request, I will break it down into two main subtasks:

1. Find out which Miami Heat player had the highest points in the 2006-2007 season.
2. Calculate the percentage change in that player's total rebounds between the 2007-2008 and 2008-2009 seasons.

Here are the tasks assigned to the respective agents:

1. WebSearchAgent : Find the Miami Heat player with the highest points in the 2006-2007 season.
2. WebSearchAgent : Find the total rebounds of the player for the 2007-2008 and 2008-2009 seasons.
3. DataAnalystAgent : Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons.

Once these tasks are completed, I will summarize the findings.
---------- ToolCallRequest

TaskResult(messages=[TextMessage(id='fd56cb60-3728-49f5-a570-3338de8b334c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 23, 13, 37, 37, 945653, tzinfo=datetime.timezone.utc), content='\nWho was the Miami Heat player with the highest points in the 2006-2007 season, \nand what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?\n', type='TextMessage'), TextMessage(id='b9cf827a-fcb8-4d38-ab12-bfccfd29bc6d', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=163, completion_tokens=174), metadata={}, created_at=datetime.datetime(2025, 7, 23, 13, 37, 43, 145973, tzinfo=datetime.timezone.utc), content="To address your request, I will break it down into two main subtasks:\n\n1. Find out which Miami Heat player had the highest points in the 2006-2007 season.\n2. Calculate the percentage change in that player's total rebounds between the 2007-2008 and 2008-2009 seasons.\n\nHere are the tasks assigned t